In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os 
import shutil 
import numpy as np

from htbam_analysis.processing import chip
from htbam_analysis.processing import experiment as exp
from htbam_analysis.processing import chipcollections as collections
from pathlib import Path 

%config InlineBackend.figure_format = 'svg'

#chip.Stamp.circlePara1Index = 30
#chip.Stamp.circlePara2Index = 15

print('Old Chamber Radius: {} pixels'.format(chip.Stamp.chamberrad))
chip.Stamp.chamberrad = 32 #33 in 2x2
print('New Chamber Radius: {} pixels'.format(chip.Stamp.chamberrad))

In [ ]:
# utility function for making reference devices

def make_reference_chip(
        reference_image: str, 
        channel: str,
        exposure: int,
        corners: tuple, 
        feature: str,
        setup_num, 
        device_num, 
        device_dimensions,
        pinlist,
        ):

    device = exp.Device(setup_num, device_num, device_dimensions, pinlist, corners)
    reference = collections.ChipQuant(device, 'Chamber_Ref')
    reference.load_file(reference_image, channel, exposure)

    if feature == 'chamber':
        reference.process(mapped_features = 'chamber', coerce_center = False)
        reference.save_summary_image(feature_type = 'chamber')
    else:
        reference.process()
        reference.save_summary_image()

    return device, reference

## 1. Establish experiment and pinlist

In [ ]:
root = '/path/to/experiment/directory/'
description = 'something_descriptive'
operator = 'whoami'
setup_num = 's#'
device_num = 'd#'
device_dimensions = (32, 56)
e = exp.Experiment(description, root, operator)

### Mock a Pinlist for Flow-On Experiments

In [ ]:
# Making a fake pinlist
#block 1 is the left side of the image, but right side of the device
block_descriptions = {1: 'protein_1', 2: 'protein_1', 3: 'protein_2', 4: 'protein_2'}
def get_block(c):
    return ((c // 8) + 1)
#creating a pin list
pinlist_dict = []
for c in range(32):
    for r in range(56):
        block = get_block(c)
        mutant = block_descriptions[block]
        pinlist_dict.append({'Indices': (c + 1, r + 1), 'MutantID': mutant})
pinlist_df = pd.DataFrame(pinlist_dict)
pinlist_path = f'{root}/data_analysis/pinlist.csv'
pinlist_df.to_csv(pinlist_path, index=False)

In [ ]:
pinlist = e.read_pinlist(f'{root}/data_analysis/pinlist.csv')
pinlist.head(5)

## 2. Button quant processing

To pick corners for your stitched button quant image, launch the corner picking application by running the following command in your terminal:

```
pick-corners /path/to/stitched/image
```

Specify the corners of your device in the application and copy the output into the notebook. Be sure that you activate your HT-BAM analysis conda environment beforehand!

In [ ]:
# specify channel and exposure used for button quant
button_quant_channel, button_quant_exposure = '2', 5

# specify path to button quant image
button_quant_path = ''

# specify corners for button quant image
# PASTE OUTPUT OF CORNER PICKER HERE
button_quant_corners = None

# specify a full path to the button quant output
button_quant_output_path = '/path/to/button_quant.csv'

# init objects for processing data
bq_device = exp.Device(setup_num, device_num, device_dimensions, pinlist, button_quant_corners)
bq_chip = collections.ChipQuant(bq_device, 'ButtonReference')
bq_chip.load_file(button_quant_path, button_quant_channel, button_quant_exposure)

# find buttons, format data into a dataframe
bq_chip.process()
button_quant_report = bq_chip.summarize()

# save the summary image and the data
button_quant_report.to_csv(button_quant_output_path)
bq_chip.save_summary_image()

## 3. Standard curve processing

Using a representative standard curve image, pick new corners using the same procedure as above.

Common failure modes (and how to address them) at this stage include:
- Poor chamber stamps across an entire image imply that corners were not picked properly. Double check that you picked good corners.
- Chamber finding will fail for chambers that are cut off from stitching artifacts. In most cases, this can be fixed by re-stitching images with an optimized rotation parameter.

In [ ]:
### DEFINE INPUTS ###

# specify channel and exposure used for standard cuver
std_channel, std_exposure = '5', 500

# specify the path to standard curve images
standard_path = ''

# specify corners for a representative standard curve image
# PASTE OUTPUT OF CORNER PICKER HERE
standard_corners = None

### CODE FOR PROCESSING ###

# init objects for processing data, load images
std_device = exp.Device(setup_num, device_num, device_dimensions, pinlist, standard_corners)
std_processor = collections.StandardSeries(std_device, std_channel)
std_processor.load_files(standard_path, std_channel, std_exposure)

# process images
std_processor.process(featuretype="chamber", coerce_center=False)

# save data and summary images
std_processor.save_summary()
std_processor.save_summary_images()

## 4. Kinetics processing


### Generate a reference image for kinetics

Finding all chambers for all images taken within a kinetic assay will take a while. Instead, we will find chambers for a single representative image and use this as a reference. 

IMPORTANT NOTE: if you exchanged pins at any point in your kinetic assay, you may need to process in batches. In some cases, exchanging pins can subtly shift the position of the device. This means that you may need to establish a new reference image with new corners for conditions after pins are exchanged.

In [ ]:
# specify channel and exposure used for kinetics
kinetics_channel, kinetics_exposure = '5', 500

# specify path to reference image to use for kinetics
kinetics_reference_image = ''

# specify corners for a representative kinetics image
# PASTE OUTPUT OF CORNER PICKER HERE
kinetics_corners = None

# make reference
kinetics_device, kinetics_reference = make_reference_chip(
    kinetics_reference_image, 
    kinetics_channel, 
    kinetics_exposure, 
    kinetics_corners, 
    'chamber',
    setup_num,
    device_num,
    device_dimensions,
    pinlist
    )

### Process kinetic images

After this is done, a `.csv.bz2` containing your processed kinetic data will be dumped into whatever directory you assign to the `kinetics_path` variable.

In [ ]:
kinetics_path = '{}/kinetics'.format(root)

# these should be UNIQUE identifiers for each kinetic image
# NOTE: you may run into issues if one identifier is a substring of another
kinetic_descriptions = ['7_8125uM_ADP',
                        # '15_6125uM_ADP',
                        # '31_25uM_ADP',
                        # '62_5uM_ADP',
                        # '125uM_ADP',
                        # '250uM_ADP',
                        # '500uM_ADP',
                        # '1000uM_ADP',
                        ]
kinetic_file_handles = kinetic_descriptions

# make processing object
# button_ref is None because we don't care about button processing here
kinetics_processor = collections.AssaySeries(kinetics_device, kinetic_descriptions, kinetics_reference.chip, button_ref=None)

# grab kinetic images
kinetics_processor.parse_kineticsFolders(kinetics_path, kinetic_file_handles, kinetic_descriptions, kinetics_channel, kinetics_exposure)

# process images and save data
kinetics_processor.process_kinetics(low_mem = False)
kinetics_processor.save_summary(outPath = kinetics_path)

## 5. Binding Assay


### Establish a reference image for binding

In [ ]:
# specify channels and exposures used for bait/prey
prey_channel, prey_exposure = '3', 5
bait_channel, bait_exposure = '2', 5

# specify path to reference image to use for kinetics
binding_reference_image = ''

# specify corners for a representative kinetics image
# PASTE OUTPUT OF CORNER PICKER HERE
binding_corners = None

# make reference
# NOTE: if using a bait image as a reference you'll need to replace the arguments
binding_device, binding_reference = make_reference_chip(
    binding_reference_image, 
    prey_channel, 
    prey_exposure, 
    binding_corners, 
    'button',
    setup_num,
    device_num,
    device_dimensions,
    pinlist
)

### Process binding images

After this is done, three `.csv.bz2` containing your processed prewash prey, postwash prey, and postwash bait data will be dumped into whatever directory you assign to the `binding_path` variable.

In [ ]:
# specify path to directory with binding images
binding_path = ''

# init processing object
binding_processor = collections.ButtonBindingSeries(
    device=binding_device, 
    button_ref=binding_reference, 
    prey_channel=prey_channel,
    prey_exposure=prey_exposure,
    bait_channel=bait_channel,
    bait_exposure=bait_exposure
    )

# grab images from bindnig experiment
binding_processor.grab_binding_images(binding_path, verbose=True)

# process images
binding_processor.process(save_summary_images=True)

# save data
# optionally provide a description string
binding_processor.save_summary(binding_path, description=None)